In [1]:
import numpy as np
import basis_set_exchange as bse


# BSE dictionary uses atomic numbers (as strings) for keys. 
# We need a quick lookup to map atomic symbols to numbers.
ATOMIC_NUMBERS = {
    'H': 1, 'HE': 2, 'LI': 3, 'BE': 4, 'B': 5, 'C': 6, 'N': 7, 'O': 8, 'F': 9, 'NE': 10,
    'NA': 11, 'MG': 12, 'AL': 13, 'SI': 14, 'P': 15, 'S': 16, 'CL': 17, 'AR': 18
}


def double_factorial(n):
    if n <= 0:
        return 1

    result = 1
    for value in range(n, 0, -2):
        result *= value
    return result

class Shell:
    def __init__(self, am, center, atom_index, exponents, coefficients, function_index):
        self.am = am
        self.center = center
        self.atom_index = atom_index
        self.function_index = function_index
        
        self.exponents = np.array(exponents)
        self.coefficients = np.array(coefficients)
        self.nprimitive = len(self.exponents)

        n_prim = self.nprimitive
        exponents = self.exponents
        coefficients = self.coefficients

        n_prim = self.nprimitive
        exponents = self.exponents
        coefficients = self.coefficients

        S = np.zeros((n_prim, n_prim))
        for i in range(n_prim):
            for j in range(n_prim):
                S[i, j] = (2.0 * np.sqrt(exponents[i] * exponents[j]) / 
                            (exponents[i] + exponents[j]))**(am + 1.5)

        # Match Psi4's common angular normalization for Cartesian shells.
        angular_norm = 1.0 / np.sqrt(double_factorial(2 * am - 1))

        self.norm_prim = angular_norm * (2.0 * exponents / np.pi)**0.75 * (4.0 * exponents)**(am / 2.0)
        self.norm_cont = 1.0 / np.sqrt(np.sum(np.outer(coefficients, coefficients) * S))

    @property
    def effective_coef(self):
        """
        Returns the fully normalized contraction coefficients array.
        """
        return self.norm_cont * self.coefficients * self.norm_prim


class Basis:
    def __init__(self):
        self.shells = []
        self.nao = 0
        self.name = ""
        self.ndocc = 0

    def nshell(self):
        return len(self.shells)

    def shell(self, idx):
        return self.shells[idx]


class Molecule:
    def __init__(self, xyz_string: str):
        """Parses a standard XYZ format string into atoms, coordinates, and charges."""
        lines = xyz_string.strip().split('\n')
        self.atoms = []
        self.coords = []
        
        # Handle optional header lines in standard XYZ files
        start_idx = 0
        if len(lines[0].split()) == 1: # First line is atom count
            start_idx = 1
            
        for line in lines[start_idx:]:
            parts = line.split()
            if not parts: continue
            self.atoms.append(parts[0])
            self.coords.append([float(x) for x in parts[1:4]])
        
        self.coords = np.array(self.coords)

        # Automatically set to close shell
        self.ndocc = len(self.atoms) * 2

        # Optionally set all to default
        self.V_nn = self.get_nuclear_repulsion()
        self.set_scf_settings()


    def set_scf_settings(self, max_iter=100, e_conv=1e-6, startup_iter=5, grad_max=1e-6, grad_rms=1e-6):
        self.max_iter = max_iter
        self.startup_iter = startup_iter
        self.e_conv = e_conv
        self.grad_max = grad_max
        self.grad_rms = grad_rms


    def get_nuclear_repulsion(self):
        """
        Calculates the nuclear-nuclear repulsion energy (V_nn) in Hartrees (a.u.).
        Assumes self.coords are in Angstroms (standard XYZ format).
        """
        V_nn = 0.0
        n_atoms = len(self.atoms)
        
        # Conversion factor for Angstroms to Bohr
        ANGSTROM_TO_BOHR = 1.8897259886
        coords_bohr = self.coords * ANGSTROM_TO_BOHR
        
        for i in range(n_atoms):
            z_i = ATOMIC_NUMBERS[self.atoms[i].upper()]
            
            for j in range(i + 1, n_atoms):
                z_j = ATOMIC_NUMBERS[self.atoms[j].upper()]
                
                dist = np.linalg.norm(coords_bohr[i] - coords_bohr[j])
                
                # Add the Coulomb repulsion for this pair
                V_nn += (z_i * z_j) / dist
                
        self.V_nn = V_nn
        return V_nn


    def build_basis(self, basis_name):        
        basis = Basis()
        
        # 1. Fetch raw dictionary from Basis Set Exchange for the unique atoms
        unique_atoms = list(set(self.atoms))
        bse_dict = bse.get_basis(basis_name, elements=unique_atoms, fmt=None)

        function_index = 0
        for atom_index, (atom, coord) in enumerate(zip(self.atoms, self.coords)):
            z_str = str(ATOMIC_NUMBERS.get(atom.upper()))
            
            if z_str not in bse_dict['elements']:
                raise ValueError(f"Basis {basis_name} does not have element {atom}")

            atom_data = bse_dict['elements'][z_str]
            for shell in atom_data['electron_shells']:
                exponents = np.array([float(e) for e in shell['exponents']])
                
                coefficients_list = shell['coefficients']
                am_list = shell['angular_momentum']

                for idx, coefficients in enumerate(coefficients_list):
                    # If there is only one am value, it applies to all coefficients in the shell 
                    if len(am_list) == 1:
                        am = am_list[0]
                    # Otherwise, BSE groups SP shells together (am = [0, 1]). 
                    elif len(am_list) == len(coefficients_list):
                        am = am_list[idx]
                    else:
                        raise ValueError(
                            f"Non-standard data organization for {basis_name}, please check. am_list: {am_list}, coefficients_list: {coefficients_list}")

                    am = int(am)
                    coefficients = np.array([float(c) for c in coefficients])

                    if len(exponents) == 0:
                        continue

                    basis.shells.append(Shell(
                        am=am,
                        center=coord,
                        atom_index=atom_index,
                        exponents=exponents,
                        coefficients=coefficients,
                        function_index=function_index
                    ))
                    
                    # Advance function index by the number of Cartesian functions: (L+1)*(L+2)/2
                    function_index += (am + 1) * (am + 2) // 2
                    
        basis.nao = function_index
        basis.name = basis_name.upper()
        return basis

In [2]:
h2o_xyz = """
O        0.000000        0.000000        0.000000
H        0.000000        0.000000        1.100000
H        1.067346        0.000000       -0.266116
"""

formaldehyde_xyz = """
C    0.000000    0.000000    0.000000
O    0.000000    0.000000    1.203000
H    0.000000    0.934000   -0.582000
H    0.000000   -0.934000   -0.582000
"""


def test_basis_set(xyz_str, basis_name):
    import psi4

    psi4.core.clean()
    psi4.core.clean_options()
    psi4.core.clean_variables()

    psi4_mol = psi4.core.Molecule.from_string(xyz_str)
    psi4.set_options({'basis': basis_name, 'puream': 0})

    wfn = psi4.core.Wavefunction.build(psi4_mol, psi4.core.get_global_option('basis'))
    psi4_basis = wfn.basisset()
    
    my_mol = Molecule(xyz_str) 
    my_basis = my_mol.build_basis(basis_name)

    print ("Psi4 total primitives")
    total_prim = 0
    for i in range(psi4_basis.nshell()):
        sh = psi4_basis.shell(i)
        total_prim += int(sh.nprimitive)
    print (total_prim)
    total_prim = 0
    print ("My total primitives")
    for i in range(my_basis.nshell()):
        sh = my_basis.shell(i)
        total_prim += int(sh.nprimitive)
    print (total_prim)

    # 1. Check total number of shells
    if psi4_basis.nshell() != my_basis.nshell():
        print(f"\n[MISMATCH] Shell count differs for {basis_name}!")
        print(f"  Psi4 has {psi4_basis.nshell()} shells, you have {my_basis.nshell()} shells.")
        return False

    # 2. Loop through and inspect each shell structurally
    for n in range(psi4_basis.nshell()):
        p4_shell = psi4_basis.shell(n)
        my_shell = my_basis.shell(n)
        
        am_map = {0: 'S', 1: 'P', 2: 'D', 3: 'F'}
        sh_type = am_map.get(p4_shell.am, f"L={p4_shell.am}")

        # Check Angular Momentum
        if p4_shell.am != my_shell.am:
            print(f"\n[MISMATCH] Shell {n} Angular Momentum mismatch!")
            print(f"  Psi4: {p4_shell.am} ({sh_type}), Mine: {my_shell.am}")
            return False

        # Check Primitive Count
        if p4_shell.nprimitive != my_shell.nprimitive:
            print(f"\n[MISMATCH] Shell {n} ({sh_type}) Primitive Count mismatch!")
            print(f"  Atom Index: {my_shell.atom_index}")
            print(f"  Psi4 primitive count: {p4_shell.nprimitive}")
            print(f"  My primitive count: {my_shell.nprimitive}")
            print(f"  Psi4 raw exponents: {[p4_shell.exp(i) for i in range(p4_shell.nprimitive)]}")
            print(f"  My raw exponents: {my_shell.exponents}")
            return False

        # Check individual Primitives within this shell
        for i in range(p4_shell.nprimitive):
            p4_exp = p4_shell.exp(i)
            p4_coef = p4_shell.coef(i)
            
            my_exp = my_shell.exponents[i]
            my_coef = my_shell.effective_coef[i]

            if not np.isclose(p4_exp, my_exp, rtol=1e-5, atol=1e-8):
                print(f"\n[MISMATCH] Shell {n} ({sh_type}), Primitive {i} Exponent mismatch!")
                print(f"  Psi4: {p4_exp}")
                print(f"  Mine: {my_exp}")
                return False

            if not np.isclose(p4_coef, my_coef, rtol=1e-5, atol=1e-8):
                print(f"\n[MISMATCH] Shell {n} ({sh_type}), Primitive {i} Coefficient mismatch!")
                print(f"  Psi4: {p4_coef}")
                print(f"  Mine: {my_coef}")
                return False

    return True
xyz_strs = [h2o_xyz, formaldehyde_xyz]
basis_names = ["sto-3g", "6-31g", "6-31g**", "cc-pvdz"]

for basis_name in basis_names:
    print(f"Testing {basis_name} ...")
    all_passed = True
    for idx, xyz_str in enumerate(xyz_strs):
        if not test_basis_set(xyz_str, basis_name):
            print(f"  Molecule {idx} failed!")
            all_passed = False
            
    if all_passed:
        print(f"Basis set {basis_name} passed !!!")
    else:
        print(f"Basis set {basis_name} failed completely.")


Testing sto-3g ...
   => Loading Basis Set <=

    Name: STO-3G
    Role: ORBITAL
    Keyword: BASIS
    atoms 1   entry O          line    81 file /localscratch/kbui/miniconda3/envs/psi4env/share/psi4/basis/sto-3g.gbs 
    atoms 2-3 entry H          line    19 file /localscratch/kbui/miniconda3/envs/psi4env/share/psi4/basis/sto-3g.gbs 

Psi4 total primitives
15
My total primitives
15
Psi4 total primitives
24
My total primitives
24
Basis set sto-3g passed !!!
Testing 6-31g ...
   => Loading Basis Set <=

    Name: STO-3G
    Role: ORBITAL
    Keyword: BASIS
    atoms 1   entry C          line    61 file /localscratch/kbui/miniconda3/envs/psi4env/share/psi4/basis/sto-3g.gbs 
    atoms 2   entry O          line    81 file /localscratch/kbui/miniconda3/envs/psi4env/share/psi4/basis/sto-3g.gbs 
    atoms 3-4 entry H          line    19 file /localscratch/kbui/miniconda3/envs/psi4env/share/psi4/basis/sto-3g.gbs 

   => Loading Basis Set <=

    Name: 6-31G
    Role: ORBITAL
    Keyword: BAS